# 4.5 — Array Functions without UDFs

**Chapter 4, section 4.4.4 "Choosing a Tier, and Removing a UDF Entirely".**

**The question this notebook answers:** the chapter says the best UDF in any codebase is the one
that gets deleted, and it uses this notebook's own code as the example. Here both versions run
side by side: the NumPy-inside-a-UDF original, preserved as the *before* picture, and the
higher-order array functions that replace it.

The two listings read almost alike, and that resemblance is the point. They differ in kind, not
in degree:

* a lambda that receives **values** runs once per row, in a Python worker process;
* a lambda that receives **columns** runs **once, on the driver, at planning time**, and returns
  a Catalyst expression that the JVM then evaluates for every row.

Telling them apart on the page is the skill this section exists to build, and the last section
of this notebook demonstrates the difference by counting the calls.

**Source.** `Notebooks old/Spark-Example-10.2-Working-With-Dataframes-Array.ipynb`, ten code
cells and no prose. Its four UDF cells are kept verbatim; everything else here is new.

Runs on a laptop in under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# The source notebook opened with `SparkContext.getOrCreate()` and a commented-out `findspark`,
# which predates Spark 2.0. One SparkSession is the entry point now.
import os, tempfile, time
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, lit
from pyspark.sql.types import ArrayType, FloatType, IntegerType

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-4.5")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| NumPy", np.__version__)

Spark 4.2.0 | NumPy 2.5.2


## The data: feature vectors as array columns

Four rows, two groups, three features each — the source notebook's own table. An array column
is how a feature vector is carried in a DataFrame, and the arithmetic below is the arithmetic of
a gradient-descent step, which is where the source notebook was heading.

In [2]:
# Let us use python Array
data2 = [(1, ([2.0, 2.0, 3.0]),),
         (1, ([2.0, 3.0, 3.0]),),
         (2, ([3.0, 2.0, 3.0]),),
         (2, ([3.0, 3.0, 3.0]),)]

df = spark.createDataFrame(data2, ["id", "features"])
df.show()
df.printSchema()

+---+---------------+
| id|       features|
+---+---------------+
|  1|[2.0, 2.0, 3.0]|
|  1|[2.0, 3.0, 3.0]|
|  2|[3.0, 2.0, 3.0]|
|  2|[3.0, 3.0, 3.0]|
+---+---------------+

root
 |-- id: long (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: double (containsNull = true)



In [3]:
# The source notebook converted the column with a UDF:
#     convert_Array_udf = udf(lambda x: x, ArrayType(FloatType(), containsNull=False))
# That UDF changes no value: it ships every array into Python and back to declare a type.
# A cast says the same thing and stays in the JVM.
df = (df.withColumn("features_array", F.col("features").cast(ArrayType(FloatType())))
        .drop("features"))
df.show()
df.printSchema()

+---+---------------+
| id| features_array|
+---+---------------+
|  1|[2.0, 2.0, 3.0]|
|  1|[2.0, 3.0, 3.0]|
|  2|[3.0, 2.0, 3.0]|
|  2|[3.0, 3.0, 3.0]|
+---+---------------+

root
 |-- id: long (nullable = true)
 |-- features_array: array (nullable = true)
 |    |-- element: float (containsNull = true)



## The before picture: NumPy inside plain UDFs

These four cells are the source notebook's, unchanged. Every array crosses into Python, is
wrapped as a NumPy array, computed on, unwrapped, and shipped back — **once per row**.

In [4]:
# get the final dataframe for gradient descent

step = 10.0
multiplyPlusStep_udf = udf(lambda x, y: [step]+np.multiply(x, y).tolist(),
                           ArrayType(FloatType(), containsNull=False))

df2 = df.withColumn('multiply', multiplyPlusStep_udf('features_array', 'features_array'))

df2.show(truncate=False)

+---+---------------+---------------------+
|id |features_array |multiply             |
+---+---------------+---------------------+
|1  |[2.0, 2.0, 3.0]|[10.0, 4.0, 4.0, 9.0]|
|1  |[2.0, 3.0, 3.0]|[10.0, 4.0, 9.0, 9.0]|
|2  |[3.0, 2.0, 3.0]|[10.0, 9.0, 4.0, 9.0]|
|2  |[3.0, 3.0, 3.0]|[10.0, 9.0, 9.0, 9.0]|
+---+---------------+---------------------+



In [5]:
# convert to 1 and 0
binary_udf = udf(lambda x: np.where(np.array(x) == 3.0, 1, 0).tolist(),
                 ArrayType(IntegerType(), containsNull=False))
df.withColumn('bin', binary_udf('features_array')).show(truncate=False)

+---+---------------+---------+
|id |features_array |bin      |
+---+---------------+---------+
|1  |[2.0, 2.0, 3.0]|[0, 0, 1]|
|1  |[2.0, 3.0, 3.0]|[0, 1, 1]|
|2  |[3.0, 2.0, 3.0]|[1, 0, 1]|
|2  |[3.0, 3.0, 3.0]|[1, 1, 1]|
+---+---------------+---------+



In [6]:
# Sum the rows
# Note: You need to convert the results back to float

# Define a UDF
sumRows_udf = udf(lambda x: float(np.sum(x)), FloatType())

# Run the UDF
df3 = df2.withColumn('total', sumRows_udf('multiply'))
df3.show(truncate=False)

+---+---------------+---------------------+-----+
|id |features_array |multiply             |total|
+---+---------------+---------------------+-----+
|1  |[2.0, 2.0, 3.0]|[10.0, 4.0, 4.0, 9.0]|27.0 |
|1  |[2.0, 3.0, 3.0]|[10.0, 4.0, 9.0, 9.0]|32.0 |
|2  |[3.0, 2.0, 3.0]|[10.0, 9.0, 4.0, 9.0]|32.0 |
|2  |[3.0, 3.0, 3.0]|[10.0, 9.0, 9.0, 9.0]|37.0 |
+---+---------------+---------------------+-----+



In [7]:
# Sum Column-Wise -- and this cell is already free of UDFs, which is worth noticing:
# the comprehension runs on the driver and builds one expression per position.
n = len(df.select('features_array').first()[0])

resultDF = df.agg(F.array(*[F.sum(F.col("features_array")[i]) for i in range(n)]).alias("sum"))
resultDF.show(truncate=False)

+------------------+
|sum               |
+------------------+
|[10.0, 10.0, 12.0]|
+------------------+



The last cell is the one the chapter singles out as already correct. `range(n)` runs **on the
driver**, at planning time, and assembles a single aggregate expression with three `sum` terms;
nothing about it touches Python at run time. It is the same expression-building pattern as the
profile in [4.7](04.07%20Data%20Validation%20and%20Profiling.ipynb), and the model for
everything below.

## The rewrite: higher-order array functions

Spark's higher-order functions take a **lambda over columns** and express the same arithmetic
natively:

| Function | What it does |
|----------|--------------|
| `transform(arr, f)` | maps `f` over an array |
| `filter(arr, f)` | selects elements (the array function, a namesake of `DataFrame.filter`) |
| `zip_with(a, b, f)` | combines two arrays element-wise through a binary function |
| `aggregate(arr, start, merge)` | folds an array to a single value |

In [8]:
step = 10.0

df2_builtin = df.withColumn(
    "multiply",
    F.concat(F.array(F.lit(step)),
             F.zip_with("features_array", "features_array",
                        lambda x, y: x * y)))

df3_builtin = df2_builtin.withColumn(
    "total", F.aggregate("multiply", F.lit(0.0), lambda acc, x: acc + x))

df3_builtin.show(truncate=False)

+---+---------------+---------------------+-----+
|id |features_array |multiply             |total|
+---+---------------+---------------------+-----+
|1  |[2.0, 2.0, 3.0]|[10.0, 4.0, 4.0, 9.0]|27.0 |
|1  |[2.0, 3.0, 3.0]|[10.0, 4.0, 9.0, 9.0]|32.0 |
|2  |[3.0, 2.0, 3.0]|[10.0, 9.0, 4.0, 9.0]|32.0 |
|2  |[3.0, 3.0, 3.0]|[10.0, 9.0, 9.0, 9.0]|37.0 |
+---+---------------+---------------------+-----+



In [9]:
# Binarization, the same way: a lambda over columns, evaluated in the JVM.
bin_builtin = df.withColumn(
    "bin", F.transform("features_array", lambda x: (x == 3.0).cast("int")))
bin_builtin.show(truncate=False)

+---+---------------+---------+
|id |features_array |bin      |
+---+---------------+---------+
|1  |[2.0, 2.0, 3.0]|[0, 0, 1]|
|1  |[2.0, 3.0, 3.0]|[0, 1, 1]|
|2  |[3.0, 2.0, 3.0]|[1, 0, 1]|
|2  |[3.0, 3.0, 3.0]|[1, 1, 1]|
+---+---------------+---------+



### The two versions are checked against each other, in code

In [10]:
# An assertion that runs is worth more than a sentence claiming equivalence, and it fails
# loudly if a later edit breaks the correspondence.
compare = (df
           .withColumn("udf_multiply",
                       multiplyPlusStep_udf('features_array', 'features_array'))
           .withColumn("builtin_multiply",
                       F.concat(F.array(F.lit(step)),
                                F.zip_with("features_array", "features_array",
                                           lambda x, y: x * y)))
           .withColumn("udf_bin", binary_udf('features_array'))
           .withColumn("builtin_bin",
                       F.transform("features_array", lambda x: (x == 3.0).cast("int"))))

compare = (compare
           .withColumn("udf_total", sumRows_udf("udf_multiply"))
           .withColumn("builtin_total",
                       F.aggregate("builtin_multiply", F.lit(0.0), lambda acc, x: acc + x)))

disagreements = compare.where(
    (F.col("udf_multiply") != F.col("builtin_multiply")) |
    (F.col("udf_bin") != F.col("builtin_bin")) |
    (F.abs(F.col("udf_total") - F.col("builtin_total")) > 1e-6)).count()
assert disagreements == 0, f"{disagreements} rows disagree"
print("all three rewrites agree with the UDFs they replace, on every row")
compare.select("features_array", "builtin_multiply", "builtin_bin", "builtin_total") \
       .show(truncate=False)

all three rewrites agree with the UDFs they replace, on every row
+---------------+---------------------+-----------+-------------+
|features_array |builtin_multiply     |builtin_bin|builtin_total|
+---------------+---------------------+-----------+-------------+
|[2.0, 2.0, 3.0]|[10.0, 4.0, 4.0, 9.0]|[0, 0, 1]  |27.0         |
|[2.0, 3.0, 3.0]|[10.0, 4.0, 9.0, 9.0]|[0, 1, 1]  |32.0         |
|[3.0, 2.0, 3.0]|[10.0, 9.0, 4.0, 9.0]|[1, 0, 1]  |32.0         |
|[3.0, 3.0, 3.0]|[10.0, 9.0, 9.0, 9.0]|[1, 1, 1]  |37.0         |
+---------------+---------------------+-----------+-------------+



## What the rewrite is worth

Four rows cannot show a transport cost. Two hundred thousand rows of fifty features can — and
the answer on this version of Spark is more interesting than "the built-in wins".

One setting has to be read before any timing is interpreted:
`spark.sql.execution.pythonUDF.arrow.enabled`. When it is on, a plain `@udf` is carried across
the process boundary as Arrow batches rather than pickled row by row, which is most of what the
chapter's hierarchy is about. So the UDF is timed **both ways**.

In [11]:
WIDE = 50
ROWS = 200_000
wide = (spark.range(ROWS)
        .withColumn("features_array",
                    F.transform(F.sequence(F.lit(1), F.lit(WIDE)),
                                lambda i: (i + F.col("id") % 7).cast("float")))
        .cache())
print(f"{wide.count():,} rows x {WIDE} features")

print("Arrow optimization for plain @udf:",
      spark.conf.get("spark.sql.execution.pythonUDF.arrow.enabled"))

def timed(label, dataframe, reps=3):
    best = None
    for _ in range(reps):
        started = time.time()
        dataframe.write.format("noop").mode("overwrite").save()
        elapsed = time.time() - started
        best = elapsed if best is None else min(best, elapsed)
    print(f"  {label:44s} {best:6.2f}s")
    return best

def udf_pipeline(multiply, total):
    return (wide.withColumn("multiply", multiply("features_array", "features_array"))
                .withColumn("total", total("multiply"))
                .select("total"))

# The same two UDFs, declared twice: once carried by Arrow, once by cloudpickle.
multiply_arrow = udf(lambda x, y: [step] + np.multiply(x, y).tolist(),
                     ArrayType(FloatType(), containsNull=False), useArrow=True)
sum_arrow = udf(lambda x: float(np.sum(x)), FloatType(), useArrow=True)
multiply_pickled = udf(lambda x, y: [step] + np.multiply(x, y).tolist(),
                       ArrayType(FloatType(), containsNull=False), useArrow=False)
sum_pickled = udf(lambda x: float(np.sum(x)), FloatType(), useArrow=False)

builtin_version = (wide
                   .withColumn("multiply",
                               F.concat(F.array(F.lit(step)),
                                        F.zip_with("features_array", "features_array",
                                                   lambda x, y: x * y)))
                   .withColumn("total",
                               F.aggregate("multiply", F.lit(0.0), lambda acc, x: acc + x))
                   .select("total"))

# Which evaluation path did each declaration actually get? 100 is the cloudpickle path,
# 101 the Arrow-optimized one.
print("evaluation types -- useArrow=True:", multiply_arrow.evalType,
      " useArrow=False:", multiply_pickled.evalType)

print("\nbest of three runs, same cached input, same output, noop sink:")
t_builtin = timed("zip_with + aggregate (no Python)", builtin_version)
t_arrow   = timed("two NumPy UDFs, Arrow transport",
                  udf_pipeline(multiply_arrow, sum_arrow))
t_pickled = timed("two NumPy UDFs, cloudpickle transport",
                  udf_pipeline(multiply_pickled, sum_pickled))
print(f"\n  relative to the built-in: Arrow UDFs {t_arrow / t_builtin:.1f}x, "
      f"pickled UDFs {t_pickled / t_builtin:.1f}x")

200,000 rows x 50 features
Arrow optimization for plain @udf: true
evaluation types -- useArrow=True: 101  useArrow=False: 100

best of three runs, same cached input, same output, noop sink:


  zip_with + aggregate (no Python)               0.24s


  two NumPy UDFs, Arrow transport                0.12s


  two NumPy UDFs, cloudpickle transport          0.13s

  relative to the built-in: Arrow UDFs 0.5x, pickled UDFs 0.6x


Two things in that table are worth reading carefully, and both are corrections to what one
would predict from the hierarchy alone.

**The two transports measured the same.** The evaluation types printed above confirm the two
declarations really did take different paths — `101` is the Arrow-optimized one and `100` the
cloudpickle one — and they cost the same here to within the noise. That is not a contradiction
of the chapter: what dominates this function is the NumPy call and the array conversion at each
end, not the bytes on the pipe. Notebook
[4.6](04.06%20UDF%20Performance%20Hierarchy.ipynb) measures the transport gap on a function
whose body is two comparisons, where transport is all there is.

**The built-in version is not guaranteed to win an element-wise array benchmark**, and on this
machine it does not. Spark's higher-order array functions are evaluated by the interpreter
rather than fused into generated code, and each of `zip_with`, `concat` and `aggregate`
allocates a new array per row; meanwhile NumPy computes fifty products in one vectorized call,
so the UDF's *arithmetic* is genuinely fast.

This is the chapter's own qualification met in practice: **the tiers differ in the cost of
moving data, not of computing with it.** The reasons to prefer the built-in form here are the
ones that do not depend on a stopwatch — it stays visible to the optimizer, it puts no Python on
the executors at all, and it cannot break when a Python or NumPy version changes underneath it.
The lambda-count below is the sharper argument, and it does not need a benchmark.

## The sharpest point: counting the lambda calls

Both versions are written with a lambda. The chapter's claim is that the two lambdas live in
different worlds — one is called once on the driver at planning time, the other once per row on
an executor. That claim is directly testable.

In [12]:
calls = []

def combine(x, y):
    calls.append(1)          # a driver-side side effect, to count invocations
    return x * y

built = wide.withColumn("m", F.zip_with("features_array", "features_array", combine))
print("lambda calls after BUILDING the expression, before any action:", len(calls))
built.select(F.size("m")).limit(1).collect()
print("lambda calls after an action over 200,000 rows              :", len(calls))
print()
print("what `combine` was called with:", "Column objects, not numbers")
print("what it returned              :", type(F.col("a") * F.col("b")).__name__)

lambda calls after BUILDING the expression, before any action: 1
lambda calls after an action over 200,000 rows              : 1

what `combine` was called with: Column objects, not numbers
what it returned              : Column


One call, before a single row had been read, and still one call after the whole table was
processed. `combine` was handed `Column` objects and returned a `Column`: a Catalyst expression,
which the JVM then evaluated for every row with no Python anywhere on the executors' path.

A UDF's lambda cannot be counted this way, and that is itself the lesson — its calls happen in a
separate Python process on an executor, once per row, where a driver-side list cannot see them.

## Conclusion

* **A UDF that only changes a type should be a cast.** The source notebook's
  `convert_Array_udf` shipped every array into Python to declare `ArrayType(FloatType())`.
* **`zip_with`, `transform` and `aggregate` express the original arithmetic natively**, and an
  assertion in this notebook confirms they agree with the UDFs they replace on every row.
* **The column-wise sum was already right.** A driver-side comprehension that builds one
  expression is not a UDF and costs nothing at run time.
* **The transport made no measurable difference to *this* function.** Arrow and cloudpickle
  timed the same, because the cost here is the NumPy call and the array conversion rather than
  the bytes moved. Check which path a UDF actually took — `evalType` says so — before
  attributing a number to a tier.
* **On element-wise array work the built-in form can lose the stopwatch**, because higher-order
  array functions are interpreted and allocate per row while NumPy vectorizes. The hierarchy is
  about transport; when transport is not the bottleneck, it does not predict the winner.
* **A lambda over columns runs once, on the driver.** Counted here: one call, before any row was
  read. A lambda over values runs once per row, in a Python worker. Two formulations that look
  interchangeable on the page sit at opposite ends of the chapter's hierarchy.

Next: [4.6](04.06%20UDF%20Performance%20Hierarchy.ipynb) measures that hierarchy tier by tier.